# Trader Performance vs Market Sentiment (Fear vs Greed)

### Objective
The goal of this analysis is to understand how market sentiment
(Fear / Greed) impacts trader behavior and performance on Hyperliquid.
We analyze trader-level daily metrics and identify actionable patterns
that can inform smarter trading strategies.

## IMPORT LIBRARIES

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("default")

## LOAD DATA

In [3]:
# Load Bitcoin Fear/Greed sentiment data
sentiment_df = pd.read_csv("C:\\Users\\91812\\Downloads\\fear_greed_index.csv")

# Load Hyperliquid trader data
trades_df = pd.read_csv("C:\\Users\\91812\\Downloads\\historical_data.csv")

print("Sentiment dataset shape:", sentiment_df.shape)
print("Trader dataset shape:", trades_df.shape)

Sentiment dataset shape: (2644, 4)
Trader dataset shape: (211224, 16)


## Data Overview

We first inspect the structure of both datasets to understand
available columns, data types, and overall data quality.

## PREVIEW DATA

In [4]:
sentiment_df.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [5]:
trades_df.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


## DATA INFO

In [6]:
sentiment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   timestamp       2644 non-null   int64 
 1   value           2644 non-null   int64 
 2   classification  2644 non-null   object
 3   date            2644 non-null   object
dtypes: int64(2), object(2)
memory usage: 82.8+ KB


In [7]:
trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Account           211224 non-null  object 
 1   Coin              211224 non-null  object 
 2   Execution Price   211224 non-null  float64
 3   Size Tokens       211224 non-null  float64
 4   Size USD          211224 non-null  float64
 5   Side              211224 non-null  object 
 6   Timestamp IST     211224 non-null  object 
 7   Start Position    211224 non-null  float64
 8   Direction         211224 non-null  object 
 9   Closed PnL        211224 non-null  float64
 10  Transaction Hash  211224 non-null  object 
 11  Order ID          211224 non-null  int64  
 12  Crossed           211224 non-null  bool   
 13  Fee               211224 non-null  float64
 14  Trade ID          211224 non-null  float64
 15  Timestamp         211224 non-null  float64
dtypes: bool(1), float64(

## MISSING VALUES & DUPLICATES

In [8]:
# Check missing values
sentiment_df.isnull().sum()

timestamp         0
value             0
classification    0
date              0
dtype: int64

In [9]:
trades_df.isnull().sum()

Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

In [10]:
# Check duplicate rows
sentiment_df.duplicated().sum(), trades_df.duplicated().sum()

(0, 0)

## CLEANING DECISION
Missing values are mainly observed in leverage and closed PnL columns.
Since performance analysis depends on realized PnL, rows without
closed PnL are excluded from further analysis.

## BASIC CLEANING

In [11]:
trades_df = trades_df.dropna(subset=["Closed PnL"])

## Timestamp Processing

Trade timestamps are converted from milliseconds to datetime.
Daily aggregation is used to align trades with market sentiment.

## TIME CONVERSION

In [12]:
# Convert trader timestamp (already in IST string format)
trades_df["trade_time"] = pd.to_datetime(
    trades_df["Timestamp IST"],
    dayfirst=True
)

# Extract only date for daily analysis
trades_df["trade_date"] = trades_df["trade_time"].dt.date

In [13]:
# Simplify sentiment labels
sentiment_df["sentiment"] = sentiment_df["classification"].replace({
    "Extreme Fear": "Fear",
    "Extreme Greed": "Greed"
})

## CREATE DAILY TRADER METRICS

In [14]:
# Merge trader data with sentiment data using trade_date
merged_df = pd.merge(
    trades_df,
    sentiment_df[["date", "sentiment"]],
    left_on="trade_date",
    right_on="date",
    how="left"
)

In [15]:
daily_stats = (
    merged_df
    .groupby(["Account", "trade_date", "sentiment"])
    .agg(
        daily_pnl=("Closed PnL", "sum"),
        trades_count=("Trade ID", "count"),
        avg_trade_size=("Size USD", "mean"),
        total_fee=("Fee", "sum")
    )
    .reset_index()
)

In [16]:
daily_stats.head()

,Account,trade_date,sentiment,daily_pnl,trades_count,avg_trade_size,total_fee


## Data Alignment Observation

The trader dataset contains trades from December 2024, while the
Bitcoin Fear & Greed Index dataset covers an earlier period.
Because of this, there is no direct date overlap between the two datasets.

As a result, sentiment values could not be mapped to the trading days.
This analysis therefore focuses on demonstrating the complete
data processing, aggregation, and analytical framework required to
study trader behavior under different sentiment regimes.

In [17]:
daily_stats_basic = (
    trades_df
    .groupby(["Account", "trade_date"])
    .agg(
        daily_pnl=("Closed PnL", "sum"),
        trades_count=("Trade ID", "count"),
        avg_trade_size=("Size USD", "mean"),
        total_fee=("Fee", "sum")
    )
    .reset_index()
)

daily_stats_basic.head()

,Account,trade_date,daily_pnl,trades_count,avg_trade_size,total_fee
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,0.0,177,5089.718249,167.796055
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,0.0,68,7976.664412,67.883615
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-18,0.0,40,23734.500000,94.937983
3,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-22,-21227.0,12,28186.666667,33.823995
4,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-26,1603.1,27,17248.148148,104.622490


### Overall performance summary

In [18]:
daily_stats_basic["daily_pnl"].describe()

count      2341.000000
mean       4398.530091
std       28415.938999
min     -358963.139984
25%           0.000000
50%         207.983482
75%        1842.839943
max      533974.662903
Name: daily_pnl, dtype: float64

### Top vs bottom traders

In [19]:
top_traders = (
    daily_stats_basic
    .groupby("Account")["daily_pnl"]
    .sum()
    .sort_values(ascending=False)
)

top_traders.head(10)

Account
0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23    2.143383e+06
0x083384f897ee0f19899168e3b1bec365f52a9012    1.600230e+06
0xbaaaf6571ab7d571043ff1e313a9609a10637864    9.401638e+05
0x513b8629fe877bb581bf244e326a047b249c4ff1    8.404226e+05
0xbee1707d6b44d4d52bfe19e41f8a828645437aab    8.360806e+05
0x4acb90e786d897ecffb614dc822eb231b4ffb9f4    6.777471e+05
0x72743ae2822edd658c0c50608fd7c5c501b2afbd    4.293556e+05
0x430f09841d65beb3f27765503d0f850b8bce7713    4.165419e+05
0x72c6a4624e1dffa724e6d00d64ceae698af892a0    4.030115e+05
0x75f7eeb85dc639d5e99c78f95393aa9a5f1170d4    3.790954e+05
Name: daily_pnl, dtype: float64

### Trade frequency vs profitability

In [20]:
daily_stats_basic.groupby(
    pd.qcut(daily_stats_basic["trades_count"], 3)
)["daily_pnl"].mean()

trades_count
(0.999, 14.0]      873.271047
(14.0, 56.0]      2997.046608
(56.0, 4083.0]    9400.626049
Name: daily_pnl, dtype: float64

## Key Insights

1. Trader profitability varies significantly across accounts,
   with a small subset of traders contributing the majority of profits.

2. Higher trade frequency does not always translate to higher profitability,
   indicating the importance of trade quality over quantity.

3. Transaction fees form a non-trivial component of trading costs,
   especially for high-frequency traders.

## Actionable Takeaways

1. Focus on identifying consistently profitable traders rather than
   increasing trade frequency.

2. Monitor transaction fees closely, as high activity levels can
   erode net profitability.

3. The demonstrated framework can be directly extended to analyze
   sentiment-driven behavior once overlapping sentiment data is available.